SALVATAGGIO E CARICAMENTO MODELLI: PERSISTENZA E DEPLOYMENT

Come uso il modello domani dentro un sistema ERP, un job schedulato o una web API?
Qui entra in gioco:
- persistenza
- serializzazione
- deployment
Un modello AI senza un salvataggio corretto è inutile in produzione.

FASE 1: TRAINING
model.fit(...)
FASE 2: SALVATAGGIO
Salvi:
- architettura
- pesi
- configurazione optimizer
- eventualmente scaler/preprocessing
FASE 3: CARICAMENTO
Richiami il modello senza riaddestrarlo.
FASE 4. DEPLOYMENT
Lo usi: in API FastAPI, in job schedulato, in un MES, in ERP, in un batch SQL/Python, in temo reale.

TensorFlow usa principalmente:
- Formato .keras: consigliato
- Vecchio formato .h5
- Formato TensorFlow SaveModel

Finchè lo script è in esecuzione, il modello vive una sorta di sogno elettrinico, se la corrente va via, la conoscenza svanisce per sempre.
Il passaggio fondamentale è la serializzazione.
Inizialmente Keras si affidava a HDF5, era comun un unico granda baule dove mettevamo tutto
Oggi le esigenze sono cambiate, oggi i modelli devono girare su smartphone o altre piattaforme, ecco perchè TensorFlow ha introdotto il SavedModel, una cartella strutturata che include il grafo computazione e i parametri.

- Il formato HDF5 è un formato legacy ideale per la semplicità di trasporto essendo un singolo file con estensione H5.
- SaveModel è il formato moderno che salva non solo i pesi ma anche l'intero grafo di esecuzione, rendendo il modello indipendente dal codice sorgente originale.
Il formato SaveModel è ottimizzato per TensorFlow Serving e per il deployment su diverse piattaforme come mobile o web.
La scelta del formato influenza la portabilità e la facilità di integrazione in sistmi di produzione.
Se vuoi che il modello funzioni su un server senza neanche avere Python installato, il SaveModel è l'unica scelta possibile.

Scegliere il formato giusto significa garantisce che il lavoro sia portabile.
Mentre H5 raggruppa tutto in un unico blocco, SaveModel separa gli asset, le variabili e i metadati, permettendo un caricamente più granulare.
Le versione più recenti di Keras (da Keras3) tendono a preferire il formato nativo di Keras che garantisce la compatibilità cross-backend tra diversi framework
SaveModel memorizza informazioni aggiuntive sulle firme delle funzioni facilitando il richiamo del modello tramite API esterne.

Cosa succese sul disco quando salviamo un SaveModel?
Cosa c'è dentro la cartella di salvataggio.
Quando salviamo un formato SaveModel, TensorFlow crea una directory contenente un file protobuf (savemodel.pd) che descrive il grafo e una sottocartella (variables) per le variabili del modello.
Questa separazione permette di aggiornare i pesi del modello senza dover modificare la struttura del grafo computazionale, senza dover ridisegnare il progetto, una caratteristica fondamentale per i sistemi scalabili.

Cosa dobbiamo salvare?
Salvare tutto o solo l'essenziale?
Durante l'addestramento il salvataggio completo può saturare il disco, spesso si preferisce salvare solo i pesi.
- Il metodo save_weights, memorizza solo i parametri del modello, richiedendo che l'architettura venga ricostruita via codice prima del caricamento.
Salvare solo i pesi è utile per il versionamento leggero del modello durante le diverse epoche di addestramento.
E' un file leggerissimo, ma per ricaricarlo dobbiamo avere lo script Python che definisce la struttura della rete.
- Il metodo save, memorizza architettura, pesi, stato dell'ottimizzatore e configurazione di compilazione in un unico passaggio.
La differenza risiede nella quantità di informazioni serializzate e nella dipendenza dal codice Python originale.
Se stai consegnando il lavoro finito al cliente, utilizza il salvataggio completo di save


Importanza del salvare lo stato dell'ottimizzatore
Salvare il modello intero include lo stato dell'ottimizzatore, permettendo di riprendere l'addestramento esattamente da dove era stato interrotto.
Keras permette anche di salvare solo la struttura in formato JSON, separandola completamente dai dati numerici dei pesi.
Durante l'addestramento si utilizzano i checkpoint per salvare automaticamente lo stato migliore del modello in base a metriche di validazione.

Ricostruzione del modello
Il processo di deserializzazione
Per caricare i soli pesi, dobbiamo prima istanziare un modello con la stessa identica struttura. Se l'architettura cambia anche solo di un neurone, il caricamente fallirà per discrepanza di forma. L'architettura che ricarichiamo deve essere identica, neurone per neurone, a quella originale.
Il salvataggio completo elimina questo rischio, poichè Keras ricostruisce l'intera gerarchia dei layer leggendo le istruzioni del file o della cartella salvata.

Inferenza e Predizione
Una volta ripristinato il modello, arriva il momento della verità, la predizone.
Utilizzo dei modelli salvati su nuovi dati.
Il fine ultimo del salvataggio è l'applicazione del modello. Una volta ricaricato in memoria, il modello è pronto per trasformare dati grezzi in previsoini accurate.

Il modello caricato è congelato, non impara più dai nuovi dati (a meno che non lo si voglia esplicitamente), si limita a trasformare gli input in output con la massima precisione possibile.

Dal file alla predizone.
- La funzione load_model ripristina istantaneamente l'oggetto Keras completo, includendo tutte le trasformazioni apprese.
Prima della predizione è fondamentale applicare ai nuovi dati lo stesso preprocessing utilizzato durante la fase di addestramento.
- Il metodo .predict restituisce tensori di output che devono essere interpretati in base alla natura del compito, sia esso regressione o classificazione.
L'inferenza è la fase in cui il modello calcola il risultato finale partendo dai parametri caricati.

Preparazione dei dati
- Dimensione del Batch: anche se vuoi predire un solo singolo valore, la rete è stata addestrata aspettandosi un blocco di dati, il modello si aspetta un input con la dimensione del batch tipicamente tramite l'espansione delle dimensioni (aggiungendo dimensione extra).
- Interpretazione Output: In caso di classificazione multiclasse, l'output sarà un vettore di probabilità la cui somma è pari a uno.
- Deployment leggero: per l'inferenza in ambienti con risorse limitate, il modello caricato può essere convertito in formati ottimizzati come TensorFlow Lite.

BEST PRACTIES
In ambiente di produzione, non si carica il modello ogni volta che arriva un cliente, si carica il modello sul server e lo si tiene pronto in RAM, questo per evitare rallentamenti durante le richieste degli utenti.
E' essenziale implementare sistemi di versionamento dei file salvati per poter tornare rapidamente a versione precedenti del modello in caso di cali di performance.

Esempio TRAINING+SALVATAGGIO

In [1]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# DATI
X = np.random.rand(1000, 3)
y = X[:,0] * 2 + X[:,1] * 3

# MODELLO
model = tf.keras.Sequential([
    layers.Input(shape=(3,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# TRAINING
model.fit(X, y, epochs=10)

# SALVATAGGIO
model.save("modello_pesi.keras")

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 7.5476 - mae: 2.5523
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.9259 - mae: 2.2279 
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.5202 - mae: 1.9119
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.2377 - mae: 1.5712
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.1591 - mae: 1.2280
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.3441 - mae: 0.9445
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8564 - mae: 0.7603
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6356 - mae: 0.6732
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5452 - mae: 0.6308
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4849 - mae: 0.5977


Questo salva:
- struttura rate
- pesi
- optimizer
- configurazione training

CARICAMENTO

In [3]:
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model("modello_pesi.keras")
X = np.array([[1, 2, 3]], dtype=np.float32)
pred = model.predict(X)
print(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
[[5.977834]]


Il modello NON ricorda il dataframe originale, ricorda solo:
- pesi neurali
- struttura metematica

Attenzione, se alleno con dati normalizzati, in produzione DEVONO passare dati normalizzati, non passare mai dati diversi.

In [ ]:
import tensorflow as tf
import keras 
import numpy as np
import os

def create_model():
    """
    Definisce un modello Sequential utilizzando la sintassi moderna.
    L'uso di keras.Input è fondamentale per inizializzare correttamente i pesi 
    prima del salvataggio.
    """
    model = keras.Sequential([
        keras.Input(shape=(10,)), # Specifica esplicitamente la forma dell'input
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    
    # Compilazione: definisce l'ottimizzatore e la funzione di perdita
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

# Inizializziamo il modello
model = create_model()

# --- 1. SALVATAGGIO NATIVO (.keras) ---
# Questa è la "best practice" assoluta nel 2025. STANDARD
# Salva tutto (architettura, pesi, stato dell'ottimizzatore) in un unico file compresso.
model.save('my_model.keras') 

# --- 2. ESPORTAZIONE PER PRODUZIONE (SavedModel) ---
# Se hai bisogno di una cartella (formato SavedModel) per TensorFlow Serving o TFLite, 
# Keras 3 richiede il metodo .export() invece di .save().
# Questo risolve il ValueError che hai ricevuto precedentemente.
model.export('my_saved_model_folder') 

# --- 3. SALVATAGGIO DEI SOLI PESI (.weights.h5) ---
# Utile se vuoi salvare solo i parametri numerici (es. durante i checkpoint).
# L'estensione .weights.h5 è lo standard attuale per evitare confusione.
model.save_weights('model_params.weights.h5')

print("Sistemi di persistenza completati con successo.")

# --- CARICAMENTO PER INFERENZA ---

# Caso A: Caricamento modello completo dal file nativo
# Non è necessario ridefinire il modello nel codice Python.
new_model = keras.models.load_model('my_model.keras')

# Caso B: Caricamento dei soli pesi in un'architettura esistente
# Richiede che il modello sia stato prima creato identico all'originale.
weights_model = create_model()
weights_model.load_weights('model_params.weights.h5')

# --- PREDIZIONE ---
# Generiamo un dato casuale (batch_size=1, features=10)
data_point = np.random.random((1, 10)).astype("float32")

# Esecuzione della predizione
# verbose=0 evita di stampare la barra di caricamento per una singola operazione
prediction = new_model.predict(data_point, verbose=0)

print(f"Risultato della predizione: {prediction[0][0]:.4f}")